> **Reference copy.** An earlier version of this project's data pipeline, kept unmodified for reference. [`pipeline_benchmark.ipynb`](pipeline_benchmark.ipynb) compares its clip processing against the current [`twitch_vod_scraper.ipynb`](../twitch_vod_scraper.ipynb); see [`README.md`](README.md) for results. It depends on TwitchDownloaderCLI, which no longer runs on current Colab, so it is not runnable as-is. The saved output below is from its last run.

# jepa project - clip scraper
This notebook processes Twitch VODs to extract highlight clips based on chat activity. It downloads the full VOD and its chat, analyzes chat message density to identify 'interesting' segments, and then generates fixed-length, normalized emote frequency vectors for each clip, alongside cropping the video around the center.

### Cell Explanations:

*   **Cell 1 - Setup and Configuration:**
    *   **Purpose**: Initializes the environment by installing necessary Python libraries (`opencv-python`, `numpy`, `tqdm`, `requests`) and downloading the `TwitchDownloaderCLI` tool. It also defines global configuration variables like `VOD_ID`, `CLIP_DURATION`, `MIN_CHAT_MESSAGES`, etc.
    *   **Libraries Used**: `os`, `sys`, `subprocess`, `json`, `shutil`, `pathlib.Path`, `urllib.request`, `stat`, `mediapipe`, `cv2`, `numpy`, `tqdm`, `requests`.

*   **Cell 2 - FaceCropper Class:**
    *   **Purpose**: Defines the `FaceCropper` class, which uses Google MediaPipe's face detection to identify the largest face in a video frame. It calculates a padded bounding box around the face and applies a smoothing algorithm to stabilize the crop across frames, creating a "cameraman" effect.
    *   **Libraries Used**: `mediapipe` (imported as `mp`), `cv2` (OpenCV), `numpy`.

*   **Cell 3 - Main Pipeline Execution:**
    *   **Purpose**: Orchestrates the main workflow:
        1.  **Download**: Uses `TwitchDownloaderCLI` to download the chat log and the full video VOD for the specified `VOD_ID`.
        2.  **Analyze Chat & Emote Mapping**: Processes the downloaded chat JSON to count messages within `CLIP_DURATION` intervals, identifying segments with `MIN_CHAT_MESSAGES` or more. It also identifies all unique emotes across all chat and creates a global `emote_to_index` mapping, saving it to `emote_mapping.json`.
        3.  **Process & Save Clips**: Iterates through the "interesting starts." For each segment:
            *   It generates a normalized emote frequency vector (a fixed-length `numpy` array based on `emote_to_index`) and saves it as a JSON file.
            *   It samples exactly 64 frames evenly across the `CLIP_DURATION` from the VOD, performs a central crop, resizes them, and saves them as a new MP4 clip. *Note: This frame-by-frame seeking from the source video can significantly increase processing time compared to sequential reading.*
            *   It extracts the audio for the `CLIP_DURATION` segment as an MP3 file.
    *   **Libraries Used**: `pathlib.Path`, `subprocess`, `json`, `cv2` (OpenCV), `numpy`, `tqdm`.

In [ ]:
import tqdm
import os
import subprocess
import json
from pathlib import Path
import requests
import cv2
import numpy as np
import sys
import urllib.request
import stat

# ==========================================
# ⚒ AUTO-SETUP & INSTALLATION (Copied from Q1tbHdDA8Vgu for robustness)
# ==========================================
def install(package):
    """Silently installs pip packages. Handles single package strings or space-separated arguments."""
    # Ensure package_args is a list for subprocess.check_call
    if isinstance(package, str):
        package_args = package.split() # Split string like "--upgrade mediapipe" into ["--upgrade", "mediapipe"]
    else:
        package_args = package # Assume it's already a list of arguments

    cmd = [sys.executable, "-m", "pip", "install"] + package_args + ["-q"]
    subprocess.check_call(cmd)

def setup_environment():
    """Checks for dependencies and installs TwitchDownloaderCLI dynamically."""
    print("⚙ Checking environment...")

    required = ['opencv-python', 'numpy', 'tqdm', 'requests']
    for lib in required:
        try:
            __import__(lib)
        except ImportError:
            print(f"   ⚒ Installing {lib}...")
            install(lib)

    cli_path = "./TwitchDownloaderCLI"
    if not os.path.exists(cli_path):
        print("   ⬇️ Fetching latest TwitchDownloaderCLI...")

        api_url = "https://api.github.com/repos/lay295/TwitchDownloader/releases/latest"
        try:
            with urllib.request.urlopen(api_url) as response:
                release_data = json.loads(response.read().decode())

            # Find the Linux-x64 zip asset in the list
            download_url = None
            for asset in release_data['assets']:
                if "Linux-x64.zip" in asset['name']:
                    download_url = asset['browser_download_url']
                    break

            if not download_url:
                raise Exception("Could not find Linux-x64 asset in latest release.")

            subprocess.run(["wget", download_url, "-O", "cli.zip"], check=True)
            subprocess.run(["unzip", "-o", "cli.zip"], check=True)

            os.chmod("TwitchDownloaderCLI", os.stat("TwitchDownloaderCLI").st_mode | stat.S_IEXEC)
            os.remove("cli.zip")
            print("   ✅ TwitchDownloaderCLI installed.")

        except Exception as e:
            print(f"   ❌ Failed to install TwitchDownloaderCLI: {e}")
            return
    else:
        print("   ✅ TwitchDownloaderCLI already present.")

    # Ensure ffmpeg is installed as it's used later
    print("Installing ffmpeg...")
    subprocess.run(["apt", "update", "-qq"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt", "install", "-qq", "ffmpeg"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("ffmpeg installed.")

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
VOD_ID = "2641901193"
OUTPUT_FOLDER = f"./dataset_{VOD_ID}"
CLIP_DURATION = 15
MIN_CHAT_MESSAGES = 40 # Minimum chat messages in a CLIP_DURATION window for an "interesting" segment.
TARGET_RES = 256
CHAT_LAG_OFFSET = 5
TWITCH_CLI_PATH = "./TwitchDownloaderCLI"

# VOD scraping process

In [ ]:


# ==========================================
# 1. BTTV HELPER
# ==========================================
def get_bttv_emotes(channel_id):
    """Fetches Global and Channel-specific BTTV emotes."""
    emotes = set()
    try:
        # 1. Global Emotes
        resp = requests.get("https://api.betterttv.net/3/cached/emotes/global")
        if resp.status_code == 200:
            for e in resp.json():
                emotes.add(e['code'])

        # 2. Channel Emotes
        if channel_id:
            resp = requests.get(f"https://api.betterttv.net/3/cached/users/twitch/{channel_id}")
            if resp.status_code == 200:
                data = resp.json()
                for e in data.get('channelEmotes', []) + data.get('sharedEmotes', []):
                    emotes.add(e['code'])
    except Exception as e:
        print(f"⚠️ BTTV Fetch Warning: {e}")

    print(f"   ✅ Loaded {len(emotes)} BTTV emote codes.")
    return emotes

# ==========================================
# 2. MAIN PIPELINE
# ==========================================
def run_pipeline():
    # Create Directories
    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    video_dir = Path(OUTPUT_FOLDER) / "video"
    audio_dir = Path(OUTPUT_FOLDER) / "audio"
    target_dir = Path(OUTPUT_FOLDER) / "target"

    for d in [video_dir, audio_dir, target_dir]:
        d.mkdir(parents=True, exist_ok=True)

    print(f"Created output directories: {video_dir}, {audio_dir}, {target_dir}")

    # --- STEP A: DOWNLOAD ---
    print(f"⬇️ Downloading Chat & Video for VOD {VOD_ID}...")

    chat_path = f"{OUTPUT_FOLDER}/chat.json"
    if not os.path.exists(chat_path):
        subprocess.run([TWITCH_CLI_PATH, "chatdownload", "--id", VOD_ID, "-o", chat_path, "-E", "-t", "20"])

    video_path = f"{OUTPUT_FOLDER}/full_vod.mp4"
    if not os.path.exists(video_path):
        subprocess.run([TWITCH_CLI_PATH, "videodownload", "--id", VOD_ID, "-o", video_path, "-q", "360p", "-t", "20"])

    # --- STEP B: ANALYZE CHAT (Fixed Parsing) ---
    print("📊 Analyzing Chat & Counting Emotes (Twitch + BTTV)...")
    with open(chat_path, 'r', encoding='utf-8') as f:
        chat_data = json.load(f)

    # Fetch BTTV Emotes
    streamer_id = chat_data.get('streamer', {}).get('id')
    bttv_emotes = get_bttv_emotes(streamer_id)

    # Buckets now store the DISTRIBUTION of emotes, not just a count
    # Format: { timestamp: { "KEKW": 5, "LUL": 2 } }
    buckets = {}

    for comment in chat_data['comments']:
        offset = int(comment['content_offset_seconds'])
        window_start = (offset // CLIP_DURATION) * CLIP_DURATION

        if window_start not in buckets: buckets[window_start] = {}

        # --- CORRECTION START: Fixed Parsing Path ---
        # Look inside comment['message']['fragments']
        msg_obj = comment.get('message', {})
        fragments = msg_obj.get('fragments', [])

        for frag in fragments:
            text = frag.get('text', '').strip()

            # Check 1: Twitch Native Emote (has 'emoticon' object)
            if frag.get('emoticon') is not None:
                buckets[window_start][text] = buckets[window_start].get(text, 0) + 1

            # Check 2: BTTV Emote (Text match)
            elif text in bttv_emotes:
                buckets[window_start][text] = buckets[window_start].get(text, 0) + 1
        # --- CORRECTION END ---

    # Filter: Keep windows where SUM of emotes >= Threshold
    interesting_starts = []
    for t, dist in buckets.items():
        if sum(dist.values()) >= MIN_CHAT_MESSAGES:
            interesting_starts.append(t)

    interesting_starts.sort()
    print(f"✅ Fzzound {len(interesting_starts)} clips where emote count >= {MIN_CHAT_MESSAGES}.")

    # --- NEW: Emote Mapping and Vector Generation ---
    print("Creating emote mapping and preparing for vector generation...")
    all_unique_emotes = set()
    for t, dist in buckets.items():
        for emote_code in dist.keys():
            all_unique_emotes.add(emote_code)

    sorted_emotes = sorted(list(all_unique_emotes))
    emote_to_index = {emote: i for i, emote in enumerate(sorted_emotes)}

    # Save emote_to_index mapping
    emote_mapping_path = Path(OUTPUT_FOLDER) / 'emote_mapping.json'
    with open(emote_mapping_path, 'w') as f:
        json.dump(emote_to_index, f)
    print(f"✅ Saved emote mapping to {emote_mapping_path}")
    print(f"Total unique emotes identified: {len(sorted_emotes)}")

    # --- STEP C: PROCESS & SAVE ---
    print("✂️ Manufacturing Dataset (Video + Audio + Target JSON)...")

    cap = cv2.VideoCapture(video_path)
    original_fps = cap.get(cv2.CAP_PROP_FPS) # Get original FPS

    # Calculate Middle Crop (As requested)
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    square_side = min(original_width, original_height)
    x_start = (original_width - square_side) // 2
    y_start = (original_height - square_side) // 2
    x_end = x_start + square_side
    y_end = y_start + square_side

    clip_counter = 0
    FRAMES_PER_CLIP = 64 # Define the fixed number of frames per clip
    output_clip_fps = FRAMES_PER_CLIP / CLIP_DURATION # Calculate output FPS to maintain 15s duration

    for start_time in tqdm.tqdm(interesting_starts, desc="Processing Clips"):
        # Filenames
        clip_id = f"clip_{VOD_ID}_{clip_counter:05d}"
        vid_file = video_dir / f"{clip_id}.mp4"
        aud_file = audio_dir / f"{clip_id}.mp3"
        tgt_file = target_dir / f"{clip_id}.json"

        clip_counter += 1

        # 1. Save Target JSON (The Normalized Emote Frequency Vector)
        if not tgt_file.exists():
            # Generate and populate the numpy array
            emote_vector = np.zeros(len(sorted_emotes))
            for emote, count in buckets[start_time].items():
                if emote in emote_to_index: # Ensure emote is in our mapping
                    emote_vector[emote_to_index[emote]] = count

            # Normalize the array
            vector_sum = emote_vector.sum()
            if vector_sum > 0:
                emote_vector = emote_vector / vector_sum

            with open(tgt_file, 'w') as f:
                json.dump(emote_vector.tolist(), f) # Save as list for JSON compatibility

        # 2. Process Video (Middle Crop)
        if not vid_file.exists():
            writer = cv2.VideoWriter(str(vid_file), cv2.VideoWriter_fourcc(*'mp4v'), output_clip_fps, (TARGET_RES, TARGET_RES))
            start_pos = int(start_time * original_fps)
            frames_captured = 0
            current_frame_idx = 0
            # Calculate how many source frames total in the 15s window
            total_source_frames = int(CLIP_DURATION * original_fps)

            while frames_captured < FRAMES_PER_CLIP:
                ret, frame = cap.read()
                if not ret: break

                # Determine if this specific frame is the one we want to keep
                # (e.g., if we want frame 0, 14, 28, etc.)
                target_frame_for_capture = int(frames_captured * (total_source_frames / FRAMES_PER_CLIP))

                if current_frame_idx == target_frame_for_capture:
                    # Middle Crop & Resize
                    crop = frame[y_start:y_end, x_start:x_end]
                    final = cv2.resize(crop, (TARGET_RES, TARGET_RES))
                    writer.write(final)
                    frames_captured += 1

                current_frame_idx += 1

            writer.release()

        # 3. Extract Audio
        if not aud_file.exists():
            subprocess.run([
                "ffmpeg", "-y", "-i", video_path, "-ss", str(start_time), "-t", str(CLIP_DURATION),
                "-vn", "-acodec", "libmp3lame", str(aud_file)
            ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    cap.release()
    print(f"\n🎉 Done! Generated {clip_counter} triplets in {OUTPUT_FOLDER}")

if __name__ == "__main__":
    setup_environment() # Ensure CLI is installed before running pipeline
    run_pipeline()

⚙ Checking environment...
   ⚒ Installing opencv-python...
   ✅ TwitchDownloaderCLI already present.
Installing ffmpeg...
ffmpeg installed.
Created output directories: dataset_2641901193/video, dataset_2641901193/audio, dataset_2641901193/target
⬇️ Downloading Chat & Video for VOD 2641901193...
📊 Analyzing Chat & Counting Emotes (Twitch + BTTV)...


FileNotFoundError: [Errno 2] No such file or directory: './dataset_2641901193/chat.json'

# Download the dataset

In [ ]:
from google.colab import files
import shutil
import os

# Define the name of the zip file (without .zip extension for make_archive)
zip_filename = f"dataset_{VOD_ID}_clips"

# Create the zip archive
# shutil.make_archive(base_name, format, root_dir)
# base_name: The name of the archive file to create, including the path
# format: The archive format (e.g., 'zip', 'tar', 'gztar', 'bztar', 'xztar')
# root_dir: The directory to archive. All paths in the archive will be relative to this directory.
#           By setting root_dir to OUTPUT_FOLDER and base_dir to '.',
#           the archive will contain the contents of OUTPUT_FOLDER directly.
print(f"Compressing '{OUTPUT_FOLDER}' into '{zip_filename}.zip'...")
shutil.make_archive(zip_filename, 'zip', os.path.dirname(OUTPUT_FOLDER), os.path.basename(OUTPUT_FOLDER))
print("Compression complete.")

# Download the zip file
print(f"Downloading {zip_filename}.zip...")
files.download(f"{zip_filename}.zip")
print("Download initiated.")